# 01 — The ewm-sm sidecar surface (`ewm-scene`)

**Question this notebook answers:** what exactly crosses the boundary
between the Python controller and the HLLSet lattice?

Every ewm-sm interaction is a subprocess call: a JSONL file goes in, one
JSON document comes out. This notebook runs each flat-frame command and
shows the raw output. Companion reference: `docs/INTERACTION_MAP.md` §1.


In [ ]:
import os, sys

# make the `lab` package importable whether jupyter starts here or in the repo root
for _p in (os.getcwd(), os.path.dirname(os.getcwd())):
    if os.path.exists(os.path.join(_p, "lab")):
        sys.path.insert(0, _p)
        PROJ = _p
        break
else:
    raise RuntimeError("run jupyter from the project root or the notebooks/ directory")

from lab import EwmScene, LayaDaemon, BonsaiClient
print("project root:", PROJ)



In [ ]:
ewm = EwmScene()
print("resolved binary:", ewm.bin)


## Frames in

The controller's lingua franca is `tid{n}` strings — Bonsai token ids
lowered to strings. Here is a three-frame "conversation": frame 2 keeps
`tid2`/`tid3`, drops `tid1`/`tid4`, and adds `tid5`.


In [ ]:
import json, os

WORK = os.path.join(PROJ, "work", "nb01")
os.makedirs(WORK, exist_ok=True)
union = os.path.join(WORK, "frames.jsonl")

frames = [
    {"id": 1, "tokens": ["tid1", "tid2", "tid3", "tid4"]},
    {"id": 2, "tokens": ["tid2", "tid3", "tid5"]},
    {"id": 3, "tokens": ["tid5", "tid6", "tid7"]},
]
with open(union, "w") as fh:
    for f in frames:
        fh.write(json.dumps(f) + "\n")
print(open(union).read())


## `ingest` — per-frame content key and popcount

`pop` is the HLLSet popcount — roughly `3 × distinct tokens` because each
token is ingested with three seeds. `key` is the content address of the
frame (`h:<sha1…>`).


In [ ]:
ewm.ingest(union, verbose=True)


## `bss` — consecutive similarity

`tau[i]` is BSSτ = `|A ∩ B| / |B|` between frames i and i+1 ("how much of
the new frame was already in the old one"). `jaccard[i]` is
`|A ∩ B| / |A ∪ B|`.


In [ ]:
ewm.bss(union, verbose=True)


## `noether` — the D/R/N decomposition

Per transition: `dp` (departed), `rp` (retained), `np` (new) popcounts,
plus three indicators:

- `ind1 = dp / |R ∪ N|` — departure dominates
- `ind3 = np / |R ∪ D|` — arrival dominates
- `ind2 = BSS(R_prev, R)` — retention-chain stability (a **similarity**,
  so low = structurally surprising; it starts at transition 2)


In [ ]:
ewm.noether(union, verbose=True)


## `materialize` — token restoration

The lattice restores each frame's tokens three ways: `ordered` (greedy
restoration of order), `set` (plain set), `beam2` (beam-2). The controller
uses `ordered` for full memory.

> Edge case: `materialize` panics on frames with 1–2 tids ("attempt to
> subtract with overflow"). `ingest`/`noether` are safe on tiny frames.


In [ ]:
ewm.materialize(union, verbose=True)


## `sidecar` — the Boolean-ring trajectory

Each frame is measured against the growing GF(2) basis *before* it is
inserted: `soft` (BSS weights over basis elements), `hard` (GF(2)
coordinates when in-span), `residual` (linear novelty), `step` (soft-key
L2 distance from the previous frame), and a jump detector over the step
series (`jumps`, threshold = mean + 3σ). `--freeze N` stops basis updates
after N frames so later soft keys share one coordinate system.


In [ ]:
ewm.sidecar(union, freeze=3, verbose=True)


## `ma` — moving-average BSS over trailing windows

`fast`/`slow` are BSS between consecutive trailing-union windows of length
`--short` / `--long`. They are the cheapest "is the conversation changing
pace?" signal.


In [ ]:
ewm.ma(union, short=1, long=2, verbose=True)
